In [1]:
import pickle
from dgllife.model.model_zoo import WeavePredictor
import dgl
import dgl.nn as nn
import dgl.function as fn
import torch.nn as tnn
import torch
import torch.optim
import torch.nn.functional as F
from torch.utils.data import random_split
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from pprint import pprint
from sklearn.preprocessing import MinMaxScaler

In [2]:
with open('graph_mean_ndatas.pickle', 'rb') as handle:
    ndatas = pickle.load(handle)

with open('graph_mean_edatas.pickle', 'rb') as handle:
    edatas = pickle.load(handle)
    
with open('graphs_mean.pickle', 'rb') as handle:
    graphs = pickle.load(handle)

In [3]:
def OHE_and_normalize(mol: str):
    n = []
    for i in range(graphs[mol].num_nodes()):
        for x in ndatas[mol]:
            if (x[0] == i):
                n.append(x[1:7]+[x[-1][0]])
                break
            elif (x[7] == i):
                n.append(x[8:14]+[x[-1][1]])
                break
    assert len(n) == graphs[mol].num_nodes()
    
    e = []
    for i, j in zip(graphs[mol].edges()[0].tolist(), graphs[mol].edges()[1].tolist()):
        for x in ndatas[mol]:
            if ((i, j) == (x[0], x[7])) or ((i, j) == (x[7], x[0])):
                e.append(x[-3:-1])
                break
        for y in edatas[mol].keys():
            if ((str(i), str(j)) == y) or ((str(j), str(i)) == y):
                e[-1].append(edatas[mol][y])
    assert len(e) == graphs[mol].num_edges()
            
    #n_df = pd.DataFrame(n)
    #encoded_column_1 = pd.get_dummies(n_df[0])
    #encoded_column_2 = pd.get_dummies(n_df[6], prefix="nei")
    #n_df = n_df.join(encoded_column_1)
    #n_df = n_df.join(encoded_column_2)
    #print(n_df)
    #df.drop(col, axis=1, inplace=True)
    # try:
    #df = df.join(encoded_column)
    # except ValueError as verr:
    #     if "overlap" in verr.__str__():
    #         df = df.merge(encoded_column, left_on="BR", right_on="BR")
    return n, e

In [4]:
sorted_keys = sorted([int(x) for x in ndatas.keys()])
n_all = []
e_all = []
for x in sorted_keys:
        x_n, x_e = OHE_and_normalize(str(x))
        n_all = n_all+x_n
        e_all = e_all+x_e


In [6]:
def save_obj_2_file(obj, fn):
    with open(fn, 'wb') as fh:
        pickle.dump(obj, fh)

In [5]:
n_df = pd.DataFrame(n_all, columns=["element", "atomic_number", "radius", "mass", "electronegativity", "hybridisation", "nei"])
e_df = pd.DataFrame(e_all)
ele_encoded = pd.get_dummies(n_df["element"], prefix="ele_")
nei_encoded = pd.get_dummies(n_df["nei"], prefix="nei")
n_df.drop("element", axis=1, inplace=True)
n_df.drop("nei", axis=1, inplace=True)
n_df = n_df.join(ele_encoded)
n_df = n_df.join(nei_encoded)
norm = MinMaxScaler().fit(n_df)
norm_n_df = norm.transform(n_df)

In [121]:
n_df.head()

,atomic_number,radius,mass,electronegativity,hybridisation,ele__B,ele__BR,ele__C,ele__CL,ele__F,...,nei_OP,nei_OS,nei_OSS,nei_P,nei_PP,nei_PS,nei_S,nei_SS,nei_SSS,nei_SSSS
0,6,70,12.00,2.55,4,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
1,6,70,12.00,2.55,4,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
2,6,70,12.00,2.55,4,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
3,6,70,12.00,2.55,4,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
4,1,25,1.01,2.20,1,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [101]:
graphs.keys()

dict_keys(['27024', '44986', '5541', '32535', '15373', '19079', '6041', '31551', '43442', '44773', '20136', '30421', '48765', '40868', '47654', '46378', '16415', '31931', '40325', '272749', '49748', '20359', '745', '38739', '29680', '20164', '33015', '43924', '38245', '48119', '41496', '7982', '38337', '4028', '47491', '47410', '48938', '113', '47243', '30203', '44886', '39278', '1938', '48876', '46390', '14506', '45729', '41678', '10060', '19043', '42115', '12031', '44552', '8026', '2418', '37232', '16813', '273068', '37950', '42275', '36468', '14291', '47823', '3743', '47398', '44660', '9753', '1923', '39700', '42798', '15419', '42998', '2126', '45308', '7984', '43942', '32090', '20662', '41054', '48972', '3811', '2666', '9719', '9367', '36341', '45298', '19012', '48258', '15952', '38291', '30210', '2112', '42340', '47221', '491', '1009', '35272', '1210', '9773', '41503', '273110', '46332', '3341', '30826', '39867', '31501', '11722', '39550', '14522', '44923', '49987', '272849', '445

In [7]:
comb_graph = dgl.batch([graphs[str(x)] for x in sorted_keys])
#nx.draw(dgl.to_networkx(graphs['21']), with_labels=True)

In [126]:
n_df.tail()

,atomic_number,radius,mass,electronegativity,hybridisation,ele__B,ele__BR,ele__C,ele__CL,ele__F,...,nei_OP,nei_OS,nei_OSS,nei_P,nei_PP,nei_PS,nei_S,nei_SS,nei_SSS,nei_SSSS
445074,8,60,16.00,3.44,2,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
445075,1,25,1.01,2.20,1,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
445076,6,70,12.00,2.55,4,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
445077,1,25,1.01,2.20,1,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
445078,1,25,1.01,2.20,1,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [86]:
save_obj_2_file(n_df, 'node_mean_dataframe.pickle')
save_obj_2_file(e_df, 'edge_mean_dataframe.pickle')
save_obj_2_file(comb_graph, 'combine_graph_mean_dataframe.pickle')

In [87]:
class GraphConv(tnn.Module):
    def __init__(self, in_feats, hid_feats, out_feats):
        super().__init__()
        self.conv1 = nn.SAGEConv(in_feats=in_feats, out_feats=hid_feats, bias=True, aggregator_type='mean')
        self.conv3 = nn.SAGEConv(in_feats=hid_feats, out_feats=out_feats, bias=True, aggregator_type='mean')
        
    def message_passing(self, g):

        g.update_all(fn.copy_e('e', 'm_e'), fn.sum('m_e', 'h_e'))
        g.update_all(fn.copy_u('h', 'm_n'), fn.sum('m_n', 'h_n'))
        g.ndata['h'] = torch.cat((g.ndata.pop('h_n'), g.ndata.pop('h_e')), 1)
        g.update_all(fn.copy_e('e', 'm_e'), fn.sum('m_e', 'h_e'))
        g.update_all(fn.copy_u('h', 'm_n'), fn.sum('m_n', 'h_n'))
        g.ndata['h'] = torch.cat((g.ndata.pop('h_n'), g.ndata.pop('h_e')), 1)
    
    def forward(self, graph, inputs):
        self.message_passing(graph)
        
        h = self.conv1(graph, inputs)
        h = F.relu(h)
        h = self.conv3(graph, h)
        with graph.local_scope():
            graph.ndata['h'] = h
            graph.apply_edges(fn.u_dot_v('h', 'h', 'score'))
            return graph.edata['score']
        

In [8]:
e_star = e_df.drop(2, axis=1)
norm_e = MinMaxScaler().fit(e_star)
norm_e_star = norm_e.transform(e_star)

In [111]:
save_obj_2_file(norm, 'norm_n.pickle')
save_obj_2_file(norm_e, 'norm_e.pickle')

In [13]:
comb_graph.ndata['h'] = torch.from_numpy(norm_n_df.astype('float32'))
comb_graph.edata['e'] = torch.from_numpy(norm_e_star.astype('float32'))
comb_graph.edata['score'] = torch.from_numpy(e_df[2].to_numpy().astype('float32'))
save_obj_2_file(comb_graph, 'combine_graph_mean.pickle')

In [15]:
dgl.distributed.partition_graph(comb_graph, "atb", 5, "atb_parted", part_method="random")

Converting to homogeneous graph takes 0.013s, peak mem: 12.453 GB
Reshuffle nodes and edges: 0.039 seconds
Split the graph: 0.116 seconds
Construct subgraphs: 0.060 seconds
Splitting the graph into partitions takes 0.216s, peak mem: 12.453 GB
part 0 has 209804 nodes and 88673 are inside the partition
part 0 has 321236 edges and 178640 are inside the partition
part 1 has 211046 nodes and 89256 are inside the partition
part 1 has 323660 edges and 179979 are inside the partition
part 2 has 210558 nodes and 89141 are inside the partition
part 2 has 322304 edges and 179199 are inside the partition
part 3 has 210979 nodes and 89317 are inside the partition
part 3 has 323318 edges and 179721 are inside the partition
part 4 has 210118 nodes and 88692 are inside the partition
part 4 has 321740 edges and 178769 are inside the partition
Save partitions: 0.563 seconds, peak memory: 12.453 GB
There are 896308 edges in the graph and 0 edge cuts for 5 partitions.


In [17]:
train_sp, val_sp, test_sp = random_split(norm_e_star, [0.5, 0.25, 0.25])
train_bin_full, train_bin, val_bin, test_bin = (np.zeros(len(comb_graph.edata['e'])) for i in range(4))
for x in train_sp.indices:
    train_bin[x] = 1
for y in val_sp.indices:
    val_bin[y] = 1
for z in test_sp.indices:
    test_bin[z] =1

for x in range(len(train_bin_full)):
    train_bin_full[x] = 1
print(train_bin_full)
print(train_bin)

[1. 1. 1. ... 1. 1. 1.]
[0. 1. 0. ... 1. 0. 0.]


In [18]:
comb_graph.edata['train_mask'] = torch.from_numpy(train_bin_full).bool()
comb_graph.edata['val_mask'] = torch.from_numpy(val_bin).bool()
comb_graph.edata['test_mask'] = torch.from_numpy(test_bin).bool()

In [21]:
dgl.distributed.partition_graph(comb_graph, "atb", 5, "atb_parted", part_method="random")

Converting to homogeneous graph takes 0.008s, peak mem: 12.453 GB
Reshuffle nodes and edges: 0.034 seconds
Split the graph: 0.114 seconds
Construct subgraphs: 0.045 seconds
Splitting the graph into partitions takes 0.193s, peak mem: 12.453 GB
part 0 has 210724 nodes and 89145 are inside the partition
part 0 has 322688 edges and 179336 are inside the partition
part 1 has 211303 nodes and 89311 are inside the partition
part 1 has 323850 edges and 179915 are inside the partition
part 2 has 209542 nodes and 88555 are inside the partition
part 2 has 320698 edges and 177927 are inside the partition
part 3 has 211193 nodes and 89253 are inside the partition
part 3 has 324000 edges and 180209 are inside the partition
part 4 has 210551 nodes and 88815 are inside the partition
part 4 has 322250 edges and 178921 are inside the partition
Save partitions: 0.568 seconds, peak memory: 12.453 GB
There are 896308 edges in the graph and 0 edge cuts for 5 partitions.


In [35]:
node_features = comb_graph.ndata['h']
edge_label = comb_graph.edata['score']
train_mask = comb_graph.edata['train_mask']


In [36]:
self_loop_g = dgl.add_self_loop(comb_graph)

In [37]:
model = GraphConv(self_loop_g.ndata['h'].shape[1], 20, 10)
optimizer = torch.optim.Adam(model.parameters())


In [90]:
device = torch.device('cuda:0')
model = model.to(device)
comb_graph = comb_graph.to(device)
edge_labels = edge_label.to(device)

In [91]:
import torch.distributed as dist
def init_process_group(world_size, rank):
    dist.init_process_group(
        backend='gloo',     # change to 'nccl' for multiple GPUs
        world_size=world_size,
        rank=rank)

In [92]:
from dgl.data import split_dataset
from dgl.dataloading import GraphDataLoader

def get_dataloaders(dataset, seed, batch_size=32):
    train_sp, val_sp, test_sp = split_dataset(dataset, frac_list=[0.8, 0.1, 0.1],
                                              shuffle=True, random_state=seed)
    train_loader = GraphDataLoader(train_set, use_ddp=True, batch_size=batch_size, shuffle=True)
    val_loader = GraphDataLoader(val_set, batch_size=batch_size)
    test_loader = GraphDataloader(test_set, batch_szie=batch_size)

    return train_loader,val_loader, test_loader

In [93]:
from torch.nn.parallel import DistributedDataParallel

def init_model(seed, model, device):
    torch.manual_seed(seed)
    model = model.to(device)
    if device.type == 'cpu':
        model = DistributedDataParallel(model)
    else:
        model = DistributedDataParallel(model, device_ids=[device], output_device=device)
        
    return model

In [94]:
def evaluate(model, dataloader, device):
    model.eval()

    total_loss = 0

    for bg, labels in dataloader:
        bg = bg.to(device)
        labels = labels.to(device)
        feats = bg.ndata.pop('h')
        with torch.no_grad():
            pred = model(bg, feats)
        total_loss += abs(pred[train_mask].flatten() - edge_label[train_mask]).mean()

    return total_loss

In [95]:
def save_model(epoch, model, optimizer, loss):
    epoch_str = str(epoch)
    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': loss}, f'{epoch_str}_model_state.pt')

In [96]:
def main(rank, world_size, dataset, seed=0):
    init_process_group(word_size, rank)
    if torch.cuda.is_available():
        device = torch.device('cude:{:d}'.format(rank))
        torch.cuda.set_device(device)
    else:
        device = torch.device('cpu')

    model = init_model(seed, model, device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    train_loader, val_loader, test_loader = get_dataloaders(dataset, seed)

    for epoch in range(1):
        model.train()
        train.loader.set_epoch(epoch)
        total_loss = 0
        for bg, labels in train_loader:
            bg = bg.to(device)
            labels = lables.to(device)
            feats = bg.ndata.pop('h')
            pred = model(bg, feats)

            loss = abs(pred.flatten() - labels).mean()
            total_loss += loss.cpu().item()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()


        loss = total_loss
        print('Loss: {:.4f}'.format(loss))

        val_loss = evaluate(model, val_loader, device)
        print('Val loss: {:.4f}'.format(val_loss))
        if epoch%1000 == 0:
            save_model(epoch, model, optimizer, total_loss)
    test_loss = evaluate(model, test_loader, device)
    print('Test loss: {:.4f}'.format(test_loss))

In [97]:
j

RuntimeError: invalid device pointer: 0x7f2f53200000

In [28]:
optimizer.param_groups[0]['lr'] = 0.01

for epoch in range(10000):
        pred = model(comb_graph, node_features)
        loss = abs(pred[train_mask].flatten() - edge_label[train_mask]).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        print(loss.item())

42317.3828125
45395.4375
42613.4375
43157.78125
44114.625
44018.8125
43267.140625
42486.421875
42561.48828125
43191.22265625
43311.27734375
42892.40234375
42422.12890625
42438.0625
42741.890625
42900.09765625
42818.875
42572.77734375
42360.515625
42431.36328125
42624.6796875
42663.41015625
42516.90625
42356.8203125
42381.03515625
42495.59375
42535.1015625
42465.45703125
42359.390625
42346.8671875
42427.640625
42456.68359375
42394.01171875
42329.4375
42356.20703125
42403.76171875
42396.4140625
42345.1328125
42326.14453125
42363.53125
42377.09375
42342.2421875
42321.6875
42345.1875
42357.0859375
42336.5078125
42319.28125
42334.43359375
42343.36328125
42325.84375
42318.265625
42330.84375
42331.66796875
42318.24609375
42318.7578125
42327.84765625
42321.7109375
42314.3125
42321.453125
42322.47265625
42313.9453125
42316.19140625
42319.91015625
42313.5078125
42313.328125
42316.36328125
42312.43359375
42312.0078125
42314.85546875
42311.38671875
42311.04296875
42312.83984375
42309.97265625
4231

KeyboardInterrupt: 

In [27]:
model.load_state_dict(torch.load('model_parms_42k.pt'))

<All keys matched successfully>

In [48]:
(edge_label[train_mask])

tensor([180040.1094, 298029.2500, 268233.4688,  ..., 185079.9219,
        198301.7969, 123835.5703])

In [29]:
abs(pred[train_mask] - edge_label[train_mask]).mean()

tensor(80652.1484, grad_fn=<MeanBackward0>)

In [37]:
torch.save(model.state_dict(), 'model_parms_42k.pt')

In [ ]:
model = GraphConv(comb_graph.ndata['h'].shape[1], 50, 20, 10)
optimizer = torch.optim.Adam(model.parameters())

model_weave = WeavePredictor(comb_graph.ndata['h'].shape[1], comb_graph.edata['e'].shape[1], 

In [42]:
print(norm_e_star)

[[0.37620873 0.20721817]
 [0.08499796 0.18207624]
 [0.0850565  0.18207624]
 ...
 [0.00174659 0.10583942]
 [0.08424243 0.17964315]
 [0.08451123 0.17680454]]
